# CropGuard Maize Disease Model: Business Intelligence and ML Workflow

**Prepared for:** Project panel  
**Project:** CropGuard, maize-only image screening  
**Scope:** Public global benchmark (PlantVillage); local field performance is not yet established.

This notebook records the reproducible steps from business question and source selection through image preparation, exploratory analysis, model training, held-out evaluation, and a CropGuard inference smoke test. Run every cell from the repository root in the project virtual environment. Training downloads data and ImageNet weights and can take substantial time on CPU.

## 1. Business understanding and BI success criteria (CRISP-DM)

**Business problem.** A farmer needs an accessible first-pass screen for a photo of a maize leaf. CropGuard should identify a healthy leaf, three selected maize disease categories, or reject an unsupported/non-maize example rather than report a tomato or bean disease.

**Decision supported.** The result is a screening prompt for further inspection, not a confirmed diagnosis or an automatic pesticide recommendation. Confirm uncertain cases with a local agricultural extension specialist.

**Stakeholders.** Farmers (usable result), researchers/agricultural officers (traceable data and class-level performance), administrators (maintainable deployment and usage reporting).

**BI / CRISP-DM stages:** business understanding -> data understanding -> data acquisition -> preparation / ETL -> exploratory analysis -> modeling -> evaluation -> deployment and monitoring.

**Measurable outputs:** per-class train/validation/test counts, no physical-leaf group overlap, overall accuracy, macro precision/recall/F1, per-class precision/recall/F1/support, confusion matrix, training curves, model/class-map artifacts, and an app preprocessing/inference smoke test. Review each metric by class; overall accuracy alone can hide a weak disease class. This notebook does not assert an accuracy target before measuring it.

| BI requirement | KPI / evidence produced here |
|---|---|
| Data provenance and governance | Dataset URL, citation, stated license, collection date at run, label mapping, and reproducible seed/configuration in the run report. |
| Data quality and coverage | Observed count by target and split; explicit handling for non-maize and unmapped maize labels; image-path deduplication. |
| Leakage control | Leaf-group IDs are partitioned before image sampling; the trainer fails on cross-partition overlap. |
| Model usefulness | Held-out accuracy plus macro and per-class precision, recall, F1, support, and confusion matrix. |
| System readiness | Exact output-index map, saved model path, and one real image processed by CropGuard's inference wrapper. |
| Operational monitoring | CropGuard prediction confidence, latency, correction feedback, and confirmed-error review are identified for the next BI cycle. |

## 2. Data source, collection method, and governance

**Primary image source:** [PlantVillage on Hugging Face](https://huggingface.co/datasets/mohanty/PlantVillage), color/RGB image archive, pinned to repository revision `9e97599868962bd0079b8db4b7f1efa9185fa1e7`, originally introduced by Mohanty, Hughes & Salathé (2016), [DOI: 10.3389/fpls.2016.01419](https://doi.org/10.3389/fpls.2016.01419). The dataset card states CC-BY-SA-3.0, 14 crop species, and 26 disease classes. Follow the dataset attribution and share-alike terms when redistributing adapted data or artifacts; check the current source terms before publication.

The dataset card describes images captured under controlled conditions. It is suitable as a reproducible benchmark, but may not represent camera variation, backgrounds, varieties, symptom stages, or disease prevalence on a particular farm. It is not a substitute for collecting and independently labeling local field images. The paper's reported accuracy is for its original multi-crop experiment, **not** this maize model.

The trainer reads the pinned source's train/test file lists and leaf-group metadata, filters the four maize labels, and samples the remaining PlantVillage classes into one `Reject` target. It extracts only selected image members from the source ZIP using HTTP byte ranges and caches them under `~/.cache/cropguard`; it does not fetch Beans or CIFAR-10. The trainer records the source revision, URL/license, run timestamp, seed, class counts, configuration, metrics, and history in `model/training_report.json`.

**Source documentation count note:** the dataset card says 54,306 images while the listed color split sizes (43,596 + 10,709) sum to 54,305. Therefore this workflow reports counts observed from the actual source lists and its own filters rather than treating the card total as a guaranteed downloaded count.

**Disease context sources:** Common rust is caused by *Puccinia sorghi*; the [Crop Protection Network](https://cropprotectionnetwork.org/encyclopedia/common-rust-of-corn) describes brick-red pustules that may darken as they age. Gray leaf spot is caused by *Cercospora zeae-maydis*; the [Crop Protection Network](https://cropprotectionnetwork.org/encyclopedia/gray-leaf-spot-of-corn) describes long, rectangular lesions bounded by veins and conditions favoring disease development. These are contextual descriptions, not labels verified for a particular uploaded photo. The model's Northern Leaf Blight class follows the dataset's label name. Local extension guidance should determine field diagnosis and management.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "train_model.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "train_model.py").is_file(), "Open/run this notebook from the CropGuard repository."
os.chdir(PROJECT_ROOT)
DATA_DIR = Path.home() / ".cache" / "cropguard"

print("Project:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("TensorFlow GPUs:", len(tf.config.list_physical_devices("GPU")))
print("Image cache:", DATA_DIR)

## 3. Data understanding: label dictionary and scope

The four in-scope maize targets come from the source label names below. Other PlantVillage crop/disease labels map to `Reject`; `Reject` is a model output, not a disease shown in the disease library. Combining many out-of-scope labels into one class is imperfect and cannot guarantee rejection of every unknown image.

In [ ]:
from train_model import CLASS_NAMES, PLANTVILLAGE_LABELS, _plant_target

label_rows = [
    ("Corn_(maize)___healthy", "Healthy"),
    ("Corn_(maize)___Northern_Leaf_Blight", "Maize Leaf Blight"),
    ("Corn_(maize)___Common_rust_", "Maize Common Rust"),
    ("Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Maize Gray Leaf Spot"),
]
label_table = pd.DataFrame(label_rows, columns=["PlantVillage source label", "CropGuard output"])
display(label_table)
print("Ordered model outputs:", CLASS_NAMES)
assert [name for _, name in label_rows] == CLASS_NAMES[:4]
assert _plant_target("Tomato___healthy") == CLASS_NAMES.index("Reject")
assert _plant_target("Corn_(maize)___healthy") == CLASS_NAMES.index("Healthy")

### Data dictionary and ETL

| Field / stage | Meaning and treatment |
|---|---|
| Source image | RGB leaf photograph from a PlantVillage class folder; non-maize folders become `Reject`. |
| Source class | Original folder/class name; mapped using the explicit whitelist above, not inferred from image pixels. |
| Leaf group | Physical-leaf identity from the source grouping map; used to prevent near-duplicate leakage across partitions. |
| Target | Integer index in `CLASS_NAMES`; mapping is exported as `model/class_indices_maize.json`. |
| Image input | EXIF-corrected RGB, resized to 224x224, CLAHE-adjusted on luminance, then scaled to [-1, 1], matching CropGuard preprocessing. |
| Partition | Deterministic 75% train / 12.5% validation / 12.5% test allocation by leaf group, seed 42. A per-class cap may downsample each partition. |

Data quality controls: deduplicate source paths, reject unmapped maize categories rather than silently assigning them to a disease, check that each target has training data, and fail if a leaf group crosses partitions. Augmentation (flip, rotation, zoom, contrast) occurs in the training model only; validation and test images are not augmented. Class weights address class-count imbalance in the loss; report the resulting per-class metrics to assess remaining bias.

## 4. Collect, prepare, train, and evaluate

The next cell runs the same trainer used for command-line model creation. It will download selected public images into the cache, load ImageNet MobileNetV2 weights if they are not cached, train a transfer-learning classifier, fine-tune the top backbone layers, and save the app model, class mapping, and evaluation report. The default cap is 1,500 examples per class per split. For a short pipeline check only, reduce the cap and epochs; do not present a tiny smoke-run score as final model performance.

In [ ]:
EPOCHS = 8
FINE_TUNE_EPOCHS = 4
BATCH_SIZE = 32
MAX_PER_CLASS = 1500  # Set to 200 for a quicker development run; 0 means no cap.

command = [
    sys.executable,
    "train_model.py",
    "--epochs", str(EPOCHS),
    "--fine-tune-epochs", str(FINE_TUNE_EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--max-per-class", str(MAX_PER_CLASS),
    "--data-dir", str(DATA_DIR),
]
print("Running:", " ".join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## 5. Business intelligence results

All counts and scores below are read from the report produced by the completed run. They are intentionally not invented or prefilled. The test set is held out from fitting and early stopping; only consult these test results once the model configuration is fixed.

In [ ]:
from app.config import Config

REPORT_PATH = Path(Config.MODEL_PATH).with_name("training_report.json")
with REPORT_PATH.open(encoding="utf-8") as report_file:
    report = json.load(report_file)

print("Source:", report["dataset_url"])
print("Source revision:", report["dataset_revision"])
print("License:", report["dataset_license"])
print("Run timestamp (UTC):", report["created_at_utc"])
print("Model setup:", report["architecture"], report["optimizer"], "lr=", report["learning_rate"])
print("Leaf-group leakage check:", report["leaf_group_leakage_check"])
print("Seed:", report["seed"], "Image size:", report["image_size"])
print("Requested epochs:", report["epochs_requested"], "Fine-tuning epochs:", report["fine_tune_epochs_requested"], "Batch size:", report["batch_size"])
print("Fine-tuning retained:", report["fine_tuning"]["selected_by_validation_loss"])

counts = pd.DataFrame(report["split_counts"]).reindex(CLASS_NAMES)
counts.index.name = "Class"
display(counts)
counts.plot(kind="bar", figsize=(12, 5), rot=25, title="Image counts by class and partition")
plt.ylabel("Images")
plt.tight_layout()
plt.show()

metrics = report["test_metrics"]
summary = pd.DataFrame({
    "Metric": ["Accuracy", "Macro precision", "Macro recall", "Macro F1"],
    "Test result": [metrics["accuracy"], metrics["macro_precision"], metrics["macro_recall"], metrics["macro_f1_score"]],
})
display(summary.style.format({"Test result": "{:.4f}"}))
per_class = pd.DataFrame(metrics["per_class"]).T
display(per_class.style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1_score": "{:.3f}"}))

In [ ]:
source_counts = pd.DataFrame(report["source_label_counts"]).fillna(0).astype(int)
source_counts.index.name = "Original PlantVillage class"
display(source_counts.sort_values("train", ascending=False))

source_root = DATA_DIR / "plantvillage-selected"
source_examples = {
    "Healthy": "Corn_(maize)___healthy",
    "Maize Leaf Blight": "Corn_(maize)___Northern_Leaf_Blight",
    "Maize Common Rust": "Corn_(maize)___Common_rust_",
    "Maize Gray Leaf Spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
}
example_paths = {
    label: next((source_root / folder).glob("*"))
    for label, folder in source_examples.items()
}
reject_path = next(
    path
    for folder in sorted(source_root.iterdir())
    if folder.is_dir() and not folder.name.startswith("Corn_(maize)___")
    for path in folder.glob("*")
)
example_paths["Reject (non-maize source example)"] = reject_path
fig, axes = plt.subplots(1, len(example_paths), figsize=(16, 4))
for ax, (label, path) in zip(axes, example_paths.items()):
    ax.imshow(plt.imread(path))
    ax.set_title(label)
    ax.axis("off")
fig.suptitle("Example images after source filtering")
fig.tight_layout()
plt.show()

In [ ]:
confusion = pd.DataFrame(
    metrics["confusion_matrix"],
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
)
display(confusion)
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(confusion.values, cmap="Blues")
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Held-out test confusion matrix")
for row in range(len(CLASS_NAMES)):
    for col in range(len(CLASS_NAMES)):
        ax.text(col, row, int(confusion.iloc[row, col]), ha="center", va="center")
fig.colorbar(image, ax=ax, label="Images")
fig.tight_layout()
plt.show()

In [ ]:
history = pd.DataFrame(report["training_history"])
display(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key in ("loss", "val_loss"):
    if key in history:
        axes[0].plot(history.index + 1, history[key], marker="o", label=key)
for key in ("accuracy", "val_accuracy"):
    if key in history:
        axes[1].plot(history.index + 1, history[key], marker="o", label=key)
axes[0].set_title("Training / validation loss")
axes[1].set_title("Training / validation accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.legend()
fig.tight_layout()
plt.show()
print("Saved report:", REPORT_PATH)
print("Model:", Config.MODEL_PATH)
print("Class map:", Config.CLASS_INDEX_PATH)

### Interpretation for the panel

- Use the confusion matrix to identify which disease classes are confused; inspect per-class recall when missed disease cases are a concern and per-class precision when false alarms are a concern.
- The `Reject` score measures performance on held-out PlantVillage non-maize categories only. It does not establish robust out-of-distribution detection for arbitrary objects, other maize diseases, or field photos.
- Class weighting and augmentation are training choices, not evidence that the benchmark is balanced or representative. Record the observed counts and class-level metrics above.
- The official PlantVillage paper's headline score is not this model's score and must not be used as a project result.
- Before field use, collect consented, appropriately licensed local maize images across farms, varieties, devices, lighting, and symptom severity; have qualified reviewers label them; freeze a separate farm/location-level test set; and report its metrics independently.

## 6. CropGuard integration verification

CropGuard and the trainer share the same 224x224 preprocessing function, MobileNetV2 input scaling, output ordering, model path, and JSON class map. This smoke check loads the saved model through the application's inference wrapper and runs one cached healthy-maize image through the app's preprocessing/inference path. It verifies artifact compatibility and latency only; it does not assert that a single sample was classified correctly.

In [ ]:
from app.ml import inference
from app.ml.preprocessing import preprocess_for_inference

with Path(Config.CLASS_INDEX_PATH).open(encoding="utf-8") as class_file:
    index_to_name = json.load(class_file)
ordered_names = [index_to_name[str(i)] for i in range(len(index_to_name))]
assert ordered_names == CLASS_NAMES, (ordered_names, CLASS_NAMES)
assert Path(Config.MODEL_PATH).is_file(), Config.MODEL_PATH

healthy_images = list((DATA_DIR / "plantvillage-selected" / "Corn_(maize)___healthy").glob("*"))
assert healthy_images, "No cached healthy maize image was found after training."
sample_path = healthy_images[0]
input_array = preprocess_for_inference(str(sample_path), Config.IMAGE_TARGET_SIZE)
predicted_name, confidence, latency_ms = inference.predict(
    input_array,
    str(sample_path),
    {"MODEL_PATH": Config.MODEL_PATH, "CLASS_INDEX_PATH": Config.CLASS_INDEX_PATH},
    ordered_names,
)
assert not inference.is_using_demo_model(), "CropGuard loaded its demo fallback instead of the trained model."
assert predicted_name in ordered_names
print({"sample": sample_path.name, "prediction": predicted_name, "confidence": round(confidence, 4), "latency_ms": latency_ms})
print("CropGuard model-loading and inference contract: PASS")

## 7. Deployment, database, and monitoring

The trainer writes the model to the `MODEL_PATH` configured in `.env`, the ordered output map to `CLASS_INDEX_PATH`, and this run's report beside the model as `training_report.json`. Verify the app uses the same environment and that each non-`Reject` class name matches a maize `DiseaseClass` row at the corresponding output index.

For an existing CropGuard database, first back it up, then run `flask --app run.py sync-maize-classes`. Unlinked non-maize disease rows are removed; rows with prediction or training-image history are retained with no current model-output index. Historical prediction records are not deleted. New databases seeded with `flask --app run.py seed-demo-data` start with the four maize disease-library entries.

Restart the Flask process after replacing model files because model inference is cached once per process. Monitor class-level farmer feedback, prediction volume, model confidence, latency, and confirmed misclassification reports. Retrain only with reviewed labels and preserve a separate, untouched field evaluation set for each release.

**Limitations / next BI iteration:** the current data is controlled-source benchmark imagery, the reject class is a coarse mixture, and this five-output classifier does not cover every maize disease, nutrient deficiency, pest, or abiotic stress. Do not prescribe chemical controls from the image result; use locally approved guidance. A panel-ready claim of local utility requires an independently collected and reviewed field test.

## References

1. Mohanty, S. P., Hughes, D. P., & Salathé, M. (2016). *Using deep learning for image-based plant disease detection*. Frontiers in Plant Science 7:1419. https://doi.org/10.3389/fpls.2016.01419
2. Mohanty et al., PlantVillage dataset card and files, https://huggingface.co/datasets/mohanty/PlantVillage (revision `9e97599868962bd0079b8db4b7f1efa9185fa1e7`; CC-BY-SA-3.0 as stated on the card; access and license checked for this workflow).
3. Crop Protection Network, *Common rust of corn*, https://cropprotectionnetwork.org/encyclopedia/common-rust-of-corn
4. Crop Protection Network, *Gray leaf spot of corn*, https://cropprotectionnetwork.org/encyclopedia/gray-leaf-spot-of-corn
5. CropGuard implementation: `train_model.py`, `app/ml/preprocessing.py`, and `app/ml/inference.py`.